# TMDB SQL Analysis

This notebook connects directly to MySQL database `TMDB` and answers all 10 analysis questions using the cleaned tables.

Set `MYSQL_HOST`, `MYSQL_PORT`, `MYSQL_USER`, `MYSQL_PASSWORD`, and `MYSQL_DATABASE` before running. The default database is `TMDB`.

The `movie_keywords` table uses a composite primary key `(movie_id, keyword_id)`, so the full many-to-many keyword relationship is preserved. Unknown budgets, revenues, and ratings are stored as SQL `NULL` and excluded naturally by aggregate functions where appropriate.

In [4]:
%pip install -q mysql-connector-python

import os
import pandas as pd
import mysql.connector

MYSQL_CONFIG = {
    'host': os.getenv('MYSQL_HOST', 'localhost'),
    'port': int(os.getenv('MYSQL_PORT', '3306')),
    'user': os.getenv('MYSQL_USER', 'root'),
    'password': os.getenv('MYSQL_PASSWORD', 'root'),
    'database': os.getenv('MYSQL_DATABASE', 'TMDB'),
}
connection = mysql.connector.connect(**MYSQL_CONFIG)
print(f'Connected to {MYSQL_CONFIG["host"]}:{MYSQL_CONFIG["port"]}/{MYSQL_CONFIG["database"]}')

def query(sql, params=None):
    cursor = connection.cursor(dictionary=True)
    cursor.execute(sql, params or ())
    rows = cursor.fetchall()
    cursor.close()
    return pd.DataFrame(rows)

Note: you may need to restart the kernel to use updated packages.
Connected to localhost:3306/TMDB



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# 1. Action movies released after 2015
q1 = query("""
    SELECT DISTINCT m.title, m.release_date
    FROM movies AS m
    JOIN movie_genres AS mg ON mg.movie_id = m.movie_id
    JOIN genres AS g ON g.genre_id = mg.genre_id
    WHERE g.genre_name = 'Action' AND m.release_date > '2015-12-31'
    ORDER BY m.release_date, m.title
""")
display(q1)

# 2. Top 10 highest-grossing movies and their genres
q2 = query("""
    SELECT m.title, m.revenue, GROUP_CONCAT(DISTINCT g.genre_name ORDER BY g.genre_name SEPARATOR ', ') AS genres
    FROM movies AS m
    LEFT JOIN movie_genres AS mg ON mg.movie_id = m.movie_id
    LEFT JOIN genres AS g ON g.genre_id = mg.genre_id
    GROUP BY m.movie_id, m.title, m.revenue
    ORDER BY m.revenue DESC
    LIMIT 10
""")
display(q2)

# 3. Average budget and revenue by genre
q3 = query("""
    SELECT g.genre_name, AVG(m.budget) AS average_budget, AVG(m.revenue) AS average_revenue
    FROM genres AS g
    JOIN movie_genres AS mg ON mg.genre_id = g.genre_id
    JOIN movies AS m ON m.movie_id = mg.movie_id
    GROUP BY g.genre_id, g.genre_name
    ORDER BY average_revenue DESC
""")
display(q3)

,title,release_date
0,The 5th Wave,2016-01-14
1,Kung Fu Panda 3,2016-01-23
2,Zoolander 2,2016-02-06
3,Deadpool,2016-02-09
4,Triple 9,2016-02-19
...,...,...
263,F1,2025-06-25
264,Superman,2025-07-09
265,The Fantastic 4: First Steps,2025-07-23
266,Predator: Badlands,2025-11-05


,title,revenue,genres
0,Avatar,2923706026,"Action, Adventure, Science Fiction"
1,Avengers: Endgame,2799439100,"Action, Adventure, Science Fiction"
2,Avatar: The Way of Water,2334484620,"Action, Adventure, Science Fiction"
3,Titanic,2264162353,"Drama, Romance"
4,Star Wars: The Force Awakens,2068223624,"Action, Adventure, Science Fiction"
5,Avengers: Infinity War,2052415039,"Action, Adventure, Science Fiction"
6,Spider-Man: No Way Home,1921206586,"Action, Adventure, Science Fiction"
7,Zootopia 2,1868208796,"Adventure, Animation, Comedy, Family, Mystery"
8,Inside Out 2,1698863816,"Adventure, Animation, Comedy, Family"
9,Jurassic World,1671537444,"Adventure, Science Fiction, Thriller"


,genre_name,average_budget,average_revenue
0,Adventure,110673597.5976,400311337.4734
1,Animation,87635507.9701,371835705.3992
2,Family,87906297.3505,353369109.6108
3,Science Fiction,96722245.9079,333648843.9599
4,Action,94227512.5063,308453177.9186
5,Fantasy,88640066.0338,302705993.3792
6,Comedy,54500177.3878,221526373.8680
7,Music,42817021.2766,201566173.2708
8,Romance,36939702.7523,174811036.0277
9,Thriller,51214051.1140,168004244.8756


In [6]:
# 4. Top 10 actors by number of movies
q4 = query("""
    SELECT actor_name, COUNT(DISTINCT movie_id) AS movie_count
    FROM `cast`
    WHERE actor_name IS NOT NULL AND actor_name <> 'Unknown'
    GROUP BY actor_name
    ORDER BY movie_count DESC, actor_name
    LIMIT 10
""")
display(q4)

# 5. Directors who directed more than 3 movies
q5 = query("""
    SELECT person_name AS director, COUNT(DISTINCT movie_id) AS movie_count
    FROM crew
    WHERE job = 'Director'
    GROUP BY person_id, person_name
    HAVING COUNT(DISTINCT movie_id) > 3
    ORDER BY movie_count DESC, director
""")
display(q5)

# 6. Top 10 most frequently used keywords
q6 = query("""
    SELECT keyword_name, COUNT(DISTINCT movie_id) AS movie_count
    FROM movie_keywords
    WHERE keyword_name IS NOT NULL AND keyword_name <> 'Unknown'
    GROUP BY keyword_id, keyword_name
    ORDER BY movie_count DESC, keyword_name
    LIMIT 10
""")
display(q6)

,actor_name,movie_count
0,Samuel L. Jackson,39
1,Brad Pitt,38
2,Robert De Niro,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Mark Wahlberg,32
6,Scarlett Johansson,32
7,Willem Dafoe,32
8,Morgan Freeman,31
9,Tom Cruise,31


,director,movie_count
0,Steven Spielberg,27
1,Ridley Scott,19
2,Tim Burton,19
3,Martin Scorsese,16
4,Michael Bay,15
...,...,...
212,Stephen Sommers,4
213,Tate Taylor,4
214,Tim Johnson,4
215,Tim Story,4


,keyword_name,movie_count
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


In [7]:
# 7. Movies above average budget but below average revenue
q7 = query("""
    SELECT title, budget, revenue
    FROM movies
    WHERE budget > (SELECT AVG(budget) FROM movies)
      AND revenue < (SELECT AVG(revenue) FROM movies)
    ORDER BY budget DESC, revenue
""")
display(q7)

# 8. Actors in a movie directed by the selected director
DIRECTOR_NAME = 'Christopher Nolan'
q8 = query("""
    SELECT DISTINCT c.actor_name, m.title
    FROM `cast` AS c
    JOIN movies AS m ON m.movie_id = c.movie_id
    JOIN crew AS cr ON cr.movie_id = c.movie_id
    WHERE cr.job = 'Director' AND cr.person_name = %s
    ORDER BY c.actor_name, m.title
""", (DIRECTOR_NAME,))
display(q8)

,title,budget,revenue
0,The Marvels,274800000,206136825
1,Red One,250000000,185700759
2,Wake Up Dead Man: A Knives Out Mystery,210000000,4000000
3,The Gray Man,200000000,454023
4,The Tomorrow War,200000000,14400000
...,...,...,...
268,Rambo III,63000000,189015611
269,Æon Flux,62000000,53321673
270,The Equalizer 2,62000000,190400157
271,Shooter,61000000,95700000


,actor_name,title
0,Aaron Eckhart,The Dark Knight
1,Al Pacino,Insomnia
2,Alon Aboutboul,The Dark Knight Rises
3,Andy Serkis,The Prestige
4,Aneurin Barnard,Dunkirk
...,...,...
105,Tom Hardy,The Dark Knight Rises
106,Tom Wilkinson,Batman Begins
107,Topher Grace,Interstellar
108,Wes Bentley,Interstellar


In [8]:
# 9. Genre with the highest average rating, considering genres with at least 20 movies
q9 = query("""
    SELECT g.genre_name, COUNT(DISTINCT m.movie_id) AS movie_count, AVG(m.vote_average) AS average_rating
    FROM genres AS g
    JOIN movie_genres AS mg ON mg.genre_id = g.genre_id
    JOIN movies AS m ON m.movie_id = mg.movie_id
    WHERE m.vote_average IS NOT NULL
    GROUP BY g.genre_id, g.genre_name
    HAVING COUNT(DISTINCT m.movie_id) >= 20
    ORDER BY average_rating DESC
    LIMIT 1
""")
display(q9)

# 10. Top 10 movies by number of keyword tags
q10 = query("""
    SELECT m.title, COUNT(DISTINCT mk.keyword_id) AS keyword_count
    FROM movies AS m
    JOIN movie_keywords AS mk ON mk.movie_id = m.movie_id
    GROUP BY m.movie_id, m.title
    ORDER BY keyword_count DESC, m.title
    LIMIT 10
""")
display(q10)

connection.close()

,genre_name,movie_count,average_rating
0,War,85,7.357576


,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54
5,The Shining,52
6,Dawn of the Planet of the Apes,51
7,How to Train Your Dragon,50
8,13 Hours: The Secret Soldiers of Benghazi,49
9,How to Train Your Dragon: The Hidden World,49
